# PyTroch Geometric
https://pytorch-geometric.readthedocs.io/en/latest/index.html

In [1]:
import os, sys
from pathlib import Path
from typing import List, Dict, Set
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv

DATA_PATH = Path("../data/FB15k-237/")
DATA_FILE = ["train.txt", "valid.txt", "test.txt"]

In [2]:
def get_triplets(path: Path) -> List:
    tsv = pd.read_csv(path, header=None, sep='\t')
    return list(tsv.itertuples(index=False, name=None))

train_triplets = get_triplets(DATA_PATH / DATA_FILE[0])
valid_triplets = get_triplets(DATA_PATH / DATA_FILE[1])
test_triplets = get_triplets(DATA_PATH / DATA_FILE[2])

triplets = train_triplets + valid_triplets + test_triplets
entity2id = {}
relation2id = {}

for h, r, t in triplets:
    entity2id.setdefault(h, len(entity2id))
    entity2id.setdefault(t, len(entity2id))
    relation2id.setdefault(r, len(relation2id))

id2entity = {v:k for k, v in entity2id.items()}
id2relation = {v:k for k, v in relation2id.items()}
num_entities = len(id2entity)
num_relations = len(id2relation)

print(f"Train Triplets: {len(train_triplets)}")
print(f"Valid Triplets: {len(valid_triplets)}")
print(f"Test Triplets: {len(test_triplets)}")
print(f"# of Entities: {num_entities}")
print(f"# of Relations: {num_relations}")

Train Triplets: 272115
Valid Triplets: 17535
Test Triplets: 20466
# of Entities: 14541
# of Relations: 237


## GCN

In [3]:
def triplets_to_tensors(triplets, entity2id, relation2id):
    h = torch.tensor([entity2id[h] for h, r, t in triplets], dtype=torch.long)
    r = torch.tensor([relation2id[r] for h, r, t in triplets], dtype=torch.long)
    t = torch.tensor([entity2id[t] for h, r, t in triplets], dtype=torch.long)
    return h, r, t

train_h, train_r, train_t = triplets_to_tensors(train_triplets, entity2id, relation2id)
valid_h, valid_r, valid_t = triplets_to_tensors(valid_triplets, entity2id, relation2id)
test_h, test_r, test_t = triplets_to_tensors(test_triplets, entity2id, relation2id)

edge_index = torch.stack([
    torch.cat([train_h, train_t]),
    torch.cat([train_t, train_h])
], dim=0)

print(f"edge_index shape: {edge_index.shape}")

edge_index shape: torch.Size([2, 544230])


In [4]:
class GCNLinkPredictor(nn.Module):
    def __init__(self, num_entities, num_relations, hidden_dim=100, out_dim=100, dropout=0.1):
        super().__init__()
        self.entity_emb = nn.Embedding(num_entities, hidden_dim)
        self.relation_emb = nn.Embedding(num_relations, out_dim)

        self.conv1 = GCNConv(hidden_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, out_dim)
        self.dropout = dropout

        nn.init.xavier_uniform_(self.entity_emb.weight)
        nn.init.xavier_uniform_(self.relation_emb.weight)

    def encode(self, edge_index):
        x = self.entity_emb.weight
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

    def score(self, z, h, r, t):
        return (z[h] * self.relation_emb(r) * z[t]).sum(dim=-1)

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GCNLinkPredictor(num_entities, num_relations).to(device)
edge_index = edge_index.to(device)
train_h, train_r, train_t = train_h.to(device), train_r.to(device), train_t.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

def sample_negatives(positive_t, num_entities):
    return torch.randint(0, num_entities, positive_t.shape, device=positive_t.device)

EPOCHS = 500

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()

    z = model.encode(edge_index)
    positive_score = model.score(z, train_h, train_r, train_t)
    
    negative_t = sample_negatives(train_t, num_entities)
    negative_score = model.score(z, train_h, train_r, negative_t)

    loss = -F.logsigmoid(positive_score).mean() - F.logsigmoid(-negative_score).mean()

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f'Epoch {epoch + 1:03d} | Loss {loss.item():.4f}')

Epoch 100 | Loss 1.0207


Epoch 200 | Loss 0.5659


Epoch 300 | Loss 0.4715


Epoch 400 | Loss 0.4297


Epoch 500 | Loss 0.4147


In [6]:
@torch.no_grad()
def evaluate(model, edge_index, h, r, t, num_entities, batch_size=128):
    model.eval()
    z = model.encode(edge_index)

    ranks = []
    for i in range(0, len(h), batch_size):
        bh, br, bt = h[i:i+batch_size], r[i:i+batch_size], t[i:i+batch_size]
        B = bh.size(0)

        query = z[bh] * model.relation_emb(br)
        scores = query @ z.t()

        true_scores = scores.gather(1, bt.unsqueeze(1))
        rank = (scores > true_scores).sum(dim=1) + 1
        ranks.append(rank)

    ranks = torch.cat(ranks).float()
    mrr = (1.0 / ranks).mean().item()
    hits1 = (ranks <= 1).float().mean().item()
    hits10 = (ranks <= 10).float().mean().item()
    return mrr, hits1, hits10

valid_h, valid_r, valid_t = valid_h.to(device), valid_r.to(device), valid_t.to(device)
mrr, h1, h10 = evaluate(model, edge_index, valid_h, valid_r, valid_t, num_entities)
print(f"Valid MRR: {mrr:.4f} | Hits@1: {h1:.4f} | Hits@10: {h10:.4f}")

Valid MRR: 0.1184 | Hits@1: 0.0680 | Hits@10: 0.2118
